In [ ]:
import pandas as pd
import sys
from upsetplot import UpSet, from_indicators
import matplotlib.pyplot as plt

sys.executable

In [ ]:
# gar ath 和sly 对multi-TIS 基因结果，过滤出CDS相关的
##########gar
df_multiTIS = pd.read_csv("table.tisseq_multiTIS.gar.csv")
# 筛选cds相关orf
df_multiTIS_filter = df_multiTIS[df_multiTIS["ORF_type"].isin(["annotated", "Overlap_uORF",  "Overlap_dORF","internal"])]
# 在此基础上判断基因是否有多TIS
gstart_counts = df_multiTIS_filter.groupby("gene_id")["ORF_gstart"].nunique()
# 找出至少有 2 个 ORF_gstart 的基因
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
# 保留这些基因的所有行
df_multi_gstart = df_multiTIS_filter[df_multiTIS_filter["gene_id"].isin(genes_multi_gstart)]
df_multi_gstart.to_csv("table.gar.CDS_related_multiTIS.csv", index=False)

print("gar cds 相关的multi-TIS基因数目：%s" %(len(df_multi_gstart['gene_id'].unique().tolist())))

##########ath
df_multiTIS = pd.read_csv("table.tisseq_multiTIS.ath.csv")
# 筛选cds相关orf
df_multiTIS_filter = df_multiTIS[df_multiTIS["ORF_type"].isin(["annotated", "Overlap_uORF",  "Overlap_dORF","internal"])]
# 在此基础上判断基因是否有多TIS
gstart_counts = df_multiTIS_filter.groupby("gene_id")["ORF_gstart"].nunique()
# 找出至少有 2 个 ORF_gstart 的基因
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
# 保留这些基因的所有行
df_multi_gstart = df_multiTIS_filter[df_multiTIS_filter["gene_id"].isin(genes_multi_gstart)]
df_multi_gstart.to_csv("table.ath.CDS_related_multiTIS.csv", index=False)

print("ath cds 相关的multi-TIS基因数目：%s" %(len(df_multi_gstart['gene_id'].unique().tolist())))

##########sly
df_multiTIS = pd.read_csv("table.tisseq_multiTIS.sly.csv")
# 筛选cds相关orf
df_multiTIS_filter = df_multiTIS[df_multiTIS["ORF_type"].isin(["annotated", "Overlap_uORF",  "Overlap_dORF","internal"])]
# 在此基础上判断基因是否有多TIS
gstart_counts = df_multiTIS_filter.groupby("gene_id")["ORF_gstart"].nunique()
# 找出至少有 2 个 ORF_gstart 的基因
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
# 保留这些基因的所有行
df_multi_gstart = df_multiTIS_filter[df_multiTIS_filter["gene_id"].isin(genes_multi_gstart)]
df_multi_gstart.to_csv("table.sly.CDS_related_multiTIS.csv", index=False)

print("sly cds 相关的multi-TIS基因数目：%s" %(len(df_multi_gstart['gene_id'].unique().tolist())))

In [ ]:
# 五个物种的 CSV 文件及物种标识
files_species = {
    "table.ath.CDS_related_multiTIS.csv": "ath",
    "table.gar.CDS_related_multiTIS.csv": "gar",
    "table.sly.CDS_related_multiTIS.csv": "sly"
}

output_fasta = "all_species_longest_ORF.fasta"

with open(output_fasta, "w") as fasta_out:
    for file, species in files_species.items():
        df = pd.read_csv(file) 
        # 统一列名
        df.columns = [c.strip() for c in df.columns]  
        
        # ORF_length 确保为数值
        df['ORF_length'] = pd.to_numeric(df['ORF_length'], errors='coerce')
        
        # 按 gene_id 取最长 ORF
        df_longest = df.loc[df.groupby('gene_id')['ORF_length'].idxmax()]

        # 写入 FASTA
        for _, row in df_longest.iterrows():
            gene_id = row['gene_id']
            orf_id = row['ORF_ID']
            aa_seq = row['AAseq']
            fasta_out.write(f">{species}|{gene_id}|{orf_id}\n")
            for i in range(0, len(aa_seq), 60):
                fasta_out.write(aa_seq[i:i+60] + "\n")

print(f"总 FASTA 文件生成完成：{output_fasta}")

## 命令行运行cd-hit，然后对输出结果进行解析columns

In [ ]:
# 解析cd-hit 结果
clstr_file = "cd-hit/clustered_ORFs.fasta.clstr"
clusters = []
cluster_id = None

with open(clstr_file) as f:
    for line in f:
        line = line.strip()
        if line.startswith('>Cluster'):
            cluster_id = line.split()[1]
        else:
            seq_id = line.split('>')[1].split('...')[0]  
            species, gene_id, orf_id = seq_id.split('|')
            clusters.append((cluster_id, species, gene_id))

df_clusters = pd.DataFrame(clusters, columns=['Cluster', 'Species', 'GeneID'])

# 构建物种 × Cluster 的二值矩阵
df_clusters['Presence'] = 1
matrix = df_clusters.pivot_table(index='Cluster', columns='Species', values='Presence', fill_value=0)
matrix.to_csv("OG_gene_presence_matrix.csv")

df_clusters.head()

In [ ]:
clusters = []
clstr_file = "cd-hit/clustered_ORFs.fasta.clstr"

with open(clstr_file) as f:
    cluster_id = None
    for line in f:
        line = line.strip()
        if line.startswith('>Cluster'):
            cluster_id = line.split()[1]
        else:
            seq_id = line.split('>')[1].split('...')[0]
            species, gene_id, orf_id = seq_id.split('|')
            clusters.append((cluster_id, species, gene_id))

df_cluster_gene = pd.DataFrame(clusters, columns=['Cluster', 'Species', 'GeneID'])
df_cluster_gene.to_csv("cluster_gene_mapping.csv", index=False)
df_cluster_gene.head()

In [ ]:
# 读取 cluster 在各物种存在情况
df_og = pd.read_csv("OG_gene_presence_matrix.csv", index_col=0)

# 读取 cluster -> gene 映射
df_map = pd.read_csv("cluster_gene_mapping.csv")

gene_presence = []

for _, row in df_map.iterrows():
    cluster = row['Cluster']
    species = row['Species']
    gene_id = row['GeneID']
    
    # 遍历所有 species，检查 cluster 是否存在
    for sp in df_og.columns:
        if df_og.loc[int(cluster), sp] > 0:
            # 如果 cluster 在该物种存在，该 gene 在该物种标记为 1
            gene_presence.append((gene_id, sp, 1))

# 构建 gene × species 二值矩阵
df_gene = pd.DataFrame(gene_presence, columns=['GeneID','Species','Presence'])
gene_matrix = df_gene.pivot_table(index='GeneID', columns='Species', values='Presence', fill_value=0)

# 保存
gene_matrix.to_csv("gene_presence_matrix.csv")

In [ ]:
# plot
df = pd.read_csv("gene_presence_matrix.csv", index_col=0)
df_bool = df.astype(bool)
upset_data = from_indicators(df_bool.columns.tolist(), df_bool)

plt.figure(figsize=(14,7))  

# 创建 UpSet 对象
upset = UpSet(
    upset_data, 
    show_counts='%d',      
    sort_by='degree',      
    sort_categories_by=None,
    min_degree=1          
)

# 可以添加自定义标题
upset.title = "Gene Presence Across Five Species"

# 绘制
upset.plot()
# 保存为 PDF
plt.savefig("plot_multiTIS_conserved.pdf", format='pdf', bbox_inches='tight')
plt.show()